In [1]:
from sklearn.cluster import KMeans
from gensim.models import Word2Vec
import time
import numpy as np
import pandas as pd
from w2v_train import review_to_wordlist
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

D:\Codding\Education\NLP\Bag of Words Meets Bags of Popcorn\w2v_train.py:21: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("html.parser"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 21 of the file D:\Codding\Education\NLP\Bag of Words Meets Bags of Popcorn\w2v_train.py. To get rid of this warning, pass the additional argument 'features="html.parser"' to the BeautifulSoup constructor.

  soup = BeautifulSoup(raw_text)
D:\Codding\Education\NLP\Bag of Words Meets Bags of Popcorn\w2v_train.py:21: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'r

Инициализация и сборка словаря Word2Vec...


2026-07-06 17:52:39,443 : INFO : PROGRESS: at sentence #90000, processed 2003713 words, keeping 48121 word types
2026-07-06 17:52:39,472 : INFO : PROGRESS: at sentence #100000, processed 2225464 words, keeping 50190 word types
2026-07-06 17:52:39,493 : INFO : PROGRESS: at sentence #110000, processed 2444322 words, keeping 52058 word types
2026-07-06 17:52:39,516 : INFO : PROGRESS: at sentence #120000, processed 2666487 words, keeping 54098 word types
2026-07-06 17:52:39,538 : INFO : PROGRESS: at sentence #130000, processed 2892314 words, keeping 55837 word types
2026-07-06 17:52:39,562 : INFO : PROGRESS: at sentence #140000, processed 3104795 words, keeping 57324 word types
2026-07-06 17:52:39,586 : INFO : PROGRESS: at sentence #150000, processed 3330431 words, keeping 59045 word types
2026-07-06 17:52:39,608 : INFO : PROGRESS: at sentence #160000, processed 3552465 words, keeping 60581 word types
2026-07-06 17:52:39,632 : INFO : PROGRESS: at sentence #170000, processed 3776047 words, 

Обучение Word2Vec (это займет пару минут)...


2026-07-06 17:52:41,574 : INFO : EPOCH 0 - PROGRESS: at 22.14% examples, 1857253 words/s, in_qsize 7, out_qsize 0
2026-07-06 17:52:42,574 : INFO : EPOCH 0 - PROGRESS: at 44.32% examples, 1861689 words/s, in_qsize 7, out_qsize 0
2026-07-06 17:52:43,588 : INFO : EPOCH 0 - PROGRESS: at 60.88% examples, 1695346 words/s, in_qsize 7, out_qsize 0
2026-07-06 17:52:44,589 : INFO : EPOCH 0 - PROGRESS: at 81.67% examples, 1706264 words/s, in_qsize 7, out_qsize 0
2026-07-06 17:52:45,433 : INFO : EPOCH 0: training on 11841450 raw words (8389827 effective words) took 4.9s, 1725996 effective words/s
2026-07-06 17:52:46,437 : INFO : EPOCH 1 - PROGRESS: at 21.40% examples, 1797246 words/s, in_qsize 7, out_qsize 0
2026-07-06 17:52:47,438 : INFO : EPOCH 1 - PROGRESS: at 44.06% examples, 1852653 words/s, in_qsize 7, out_qsize 0
2026-07-06 17:52:48,441 : INFO : EPOCH 1 - PROGRESS: at 66.69% examples, 1864662 words/s, in_qsize 7, out_qsize 0
2026-07-06 17:52:49,442 : INFO : EPOCH 1 - PROGRESS: at 89.38% exa

Word2Vec успешно обучен и сохранен!


In [2]:
train = pd.read_csv("data/labeledTrainData.tsv", header=0,
                    delimiter="\t", quoting=3)

test = pd.read_csv("data/testData.tsv", header=0,
                    delimiter="\t", quoting=3)

In [3]:
start = time.time() # Start time

model_w2v = Word2Vec.load("models/300features_40minwords_10context_w2v")

word_vectors = model_w2v.wv.vectors
num_clusters = int(word_vectors.shape[0] / 5)

kmeans_clustering = KMeans(n_clusters = num_clusters)
idx = kmeans_clustering.fit_predict(word_vectors)

end = time.time()
elapsed = end - start
print("Time taken for K Means clustering: ", elapsed, "seconds.")

word_centroid_map = dict(zip(model_w2v.wv.key_to_index, idx))

2026-07-06 17:53:03,661 : INFO : loading Word2Vec object from models/300features_40minwords_10context_w2v
2026-07-06 17:53:03,674 : INFO : loading wv recursively from models/300features_40minwords_10context_w2v.wv.* with mmap=None
2026-07-06 17:53:03,674 : INFO : setting ignored attribute cum_table to None
2026-07-06 17:53:03,727 : INFO : Word2Vec lifecycle event {'fname': 'models/300features_40minwords_10context_w2v', 'datetime': '2026-07-06T17:53:03.727152', 'gensim': '4.4.0', 'python': '3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26200-SP0', 'event': 'loaded'}


Time taken for K Means clustering:  26.144740343093872 seconds.


In [4]:
for cluster in range(0, 20):
    print("\nCluster %d" % cluster)

    words = []
    for word, cluster_num in word_centroid_map.items():
        if cluster_num == cluster:
            words.append(word)

    print(words)


Cluster 0
['nature', 'moral', 'spiritual', 'morality', 'inherent']

Cluster 1
['bean']

Cluster 2
['tour', 'march', 'royal', 'records', 'august', 'jury', 'cuban', 'convention', 'gathering', 'informer', 'presidential', 'division', 'apollo', 'council', 'bomber', 'preparation', 'suite', 'unprecedented']

Cluster 3
['celebrity', 'designer', 'mega', 'peruvian', 'specialty']

Cluster 4
['polish', 'overt', 'expressionist']

Cluster 5
['kills', 'shoots', 'eats', 'drinks', 'puppy', 'stabs', 'shouts', 'ate', 'slapping', 'looses']

Cluster 6
['portray', 'express', 'achieve', 'accomplish', 'demonstrate', 'define', 'construct']

Cluster 7
['norman', 'hughes', 'bennett', 'doug', 'currie', 'hartman', 'roland', 'hasselhoff', 'garson', 'brody']

Cluster 8
['action', 'factor', 'surprises', 'scares', 'thrills', 'thrill', 'shocks']

Cluster 9
['flat', 'bland', 'shallow', 'forgettable', 'clumsy', 'stilted', 'stale', 'lackluster', 'lifeless', 'tolerable', 'pedestrian', 'vapid', 'lethargic']

Cluster 10
['y

In [5]:
def create_bag_of_centroids(wordlist, word_centroid_map):
    num_centroids = max(word_centroid_map.values()) + 1
    bag_of_centroids = np.zeros( num_centroids, dtype="float32" )

    for word in wordlist:
        if word in word_centroid_map:
            index = word_centroid_map[word]
            bag_of_centroids[index] += 1

    return bag_of_centroids

In [6]:
clean_train_reviews = []
for review in train["review"]:
    clean_train_reviews.append(review_to_wordlist(review, remove_stopwords=True))

clean_test_reviews = []
for review in test["review"]:
    clean_test_reviews.append(review_to_wordlist(review, remove_stopwords=True))

D:\Codding\Education\NLP\Bag of Words Meets Bags of Popcorn\w2v_train.py:21: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("html.parser"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 21 of the file D:\Codding\Education\NLP\Bag of Words Meets Bags of Popcorn\w2v_train.py. To get rid of this warning, pass the additional argument 'features="html.parser"' to the BeautifulSoup constructor.

  soup = BeautifulSoup(raw_text)


In [7]:
train_centroids = np.zeros((train["review"].size, num_clusters), dtype="float32" )

counter = 0
for review in clean_train_reviews:
    train_centroids[counter] = create_bag_of_centroids(review, word_centroid_map)
    counter += 1

test_centroids = np.zeros((test["review"].size, num_clusters), dtype="float32")

counter = 0
for review in clean_test_reviews:
    test_centroids[counter] = create_bag_of_centroids(review, word_centroid_map)
    counter += 1

In [8]:
forest = RandomForestClassifier(n_estimators = 100)

print("Fitting a random forest to labeled training data...")
forest = forest.fit(train_centroids,train["sentiment"])
result = forest.predict(test_centroids)

output = pd.DataFrame(data={"id":test["id"], "sentiment":result})
output.to_csv("results/BagOfCentroids.csv", index=False, quoting=3)

Fitting a random forest to labeled training data...


# _Metrics_

In [9]:
cv_model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)

accuracy_scores = cross_val_score(cv_model, train_centroids, train['sentiment'], cv=5, scoring='accuracy')
print(f"Кросс-валидация Accuracy: {accuracy_scores.mean() * 100:.2f}% (разброс: +/- {accuracy_scores.std() * 100:.2f}%)")

roc_auc_scores = cross_val_score(cv_model, train_centroids, train['sentiment'], cv=5, scoring='roc_auc')
print(f"Кросс-валидация ROC AUC:  {roc_auc_scores.mean() * 100:.2f}%")

Кросс-валидация Accuracy: 84.10% (разброс: +/- 0.56%)
Кросс-валидация ROC AUC:  91.45%
